In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model
)


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 只加载 mouse6_021322_natural_image_001 这个session
target_session_name = 'mouse6_021322_natural_image_001'

recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{target_session_name}.ns4')
recording_recorded = recording_raw.remove_channels(["98", '31', '32'])

probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])
print(recording_cmr)

output_folder = '/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/'
combined_output_base = output_folder

sampling_frequency = recording_cmr.get_sampling_frequency()

segment_sample_ranges = {} 
segment_num_samples_dict = {} 
session_names = [] 

session_name = target_session_name
session_names.append(session_name)



ChannelSliceRecording: 30 channels - 10000.0Hz - 1 segments - 40,000,100 samples 
                       4,000.01s (1.11 hours) - int16 dtype - 2.24 GiB


In [3]:
probe = recording_cmr.get_probe()
probe_df = probe.to_dataframe()
all_channel_ids = probe_df['contact_ids'].astype(str).tolist()
cliques = [
    CliqueInfo(
        clique_id=0,
        device_channel_indices=list(range(len(all_channel_ids))),
        contact_ids=all_channel_ids,
        center=(probe_df['x'].mean(), probe_df['y'].mean())
    )
]

for clique in cliques:
    clique_id = clique.clique_id
    session_name = target_session_name
    print(f"\n处理 Session ({session_name})...")
    
    session_data_folder = f'{combined_output_base}/clique_{clique_id}/{session_name}'
    neuron_inf_path = f'{session_data_folder}/neuron_inf.pickle'
    gt_detect_array_path = f'{session_data_folder}/gt_detect_array.csv'
    
    if not os.path.exists(neuron_inf_path) or not os.path.exists(gt_detect_array_path):
        print(f"  警告: {session_data_folder} 下没有找到数据文件，跳过")
        continue
    
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf_dict = pickle.load(f)
    gt_detect_array = pd.read_csv(gt_detect_array_path)
    
    # 转换为DataFrame
    neuron_inf_session = neuron_inf_dict_to_dataframe(neuron_inf_dict)
    
    print(f"  Neurons: {len(neuron_inf_session)}")
    print(f"  Spikes: {len(gt_detect_array)}")
    
    recording_clique = get_recording_clique(recording_cmr, clique)
    print(f"  Recording clique channels: {len(recording_clique.get_channel_ids())}")
    
    # 准备训练数据
    clique_save_dir = f'{combined_output_base}/clique_{clique_id}/{session_name}'
    train_data_dir = prepare_training_data(
        recording_f=recording_clique,
        gt_detect_array=gt_detect_array,
        neuron_inf=neuron_inf_session,
        save_dir=clique_save_dir,
        duration_seconds=4000,
        thr_min=5,
        thr_max=35,
        distance=3,
        wlen=5,
        prominence=20,
        left_sample=10,
        right_sample=20,
        max_firing_channel=10
    )
    
    n_channels = recording_clique.get_num_channels()
    n_repeats = 1
    
    for repeat_idx in range(1, n_repeats + 1):
        print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
        model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
        
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=model_save_dir,
            n_channels=n_channels,
            left_sample=10,
            right_sample=20,
            epochs=20,
            batch_size=512,
            device=None,
            early_stopping=True,
            patience=5,
            min_delta=0.0,
            use_focal_loss=True,
            focal_gamma=2.0
        )
            
    print(f"  Clique {clique_id}, Session {session_name} 所有重复训练完成!")


处理 Session (mouse6_021322_natural_image_001)...
  Neurons: 39
  Spikes: 1032661
  Recording clique channels: 30
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 30
Data shape: (40000000, 30) (clique channels)
Building detect_array...
Number of detected spikes: 2876868
去重: 移除了296503个spikes（保留幅值更大的channel上的spike）
去重前: 2876868个spikes, 去重后: 2580365个spikes
Number of detected spikes after deduplication: 2580365

### 2. Load Ground Truth and Match
GT匹配统计: 978823/1032657 GT spikes被检测到 (召回率: 0.9479)
补全未匹配的GT spikes: 53834个
  添加了53833个未匹配的GT spikes到detect_array
  为53833个未匹配的GT spikes设置了标签
补全后GT覆盖率: 1032656/1032657 GT spikes (1.0000)
  - 检测到的GT spikes: 978823
  - 补全的GT spikes: 53833
  - 总spike数量: 2634198 (原始: 2580365, 新增: 53833)

### 3. Extract Waveforms


Extracting waveforms: 100%|██████████| 30/30 [00:51<00:00,  1.72s/it]


Waveform extraction completed!
waveform shape: (2634197, 30, 30)

### 4. Save Training Data
Save directory: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/clique_0/mouse6_021322_natural_image_001/train_data
  ✓ neuron_mapping.pkl saved
Saving data...
  ✓ X_waveform.pkl saved
  ✓ Y_spike_id.pkl saved
  ✓ Y_spike_id_noise.pkl saved
  ✓ X_spiketrain_time.pkl saved

All data saved to: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full/clique_0/mouse6_021322_natural_image_001/train_data
Data statistics:
  - Total spike count: 2634197
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 39
  - Noise spike count: 1601542
  - Valid spike count: 1032655

  ===== 重复训练 1/1 =====
Using device: cuda
Create dataset...
Auto-extracting keep_id from data
Dataset loaded:
  - Total samples: 2634197
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 39
  - Noise samples: 1601542.0
  

Training: 100%|██████████| 4116/4116 [00:25<00:00, 161.05it/s]


epoch : 1/20, detection loss = 36.354387, classification loss = 382.300030


...

epoch : 10/20, val detection loss = 18.934017, classification loss = 18.380170
epoch : 10/20, val acc noise = 0.9453, val acc label = 0.9764
epoch : 11/20


Training: 100%|██████████| 4116/4116 [00:26<00:00, 158.01it/s]


epoch : 11/20, detection loss = 8.487841, classification loss = 9.343162


Validation: 100%|██████████| 1029/1029 [00:04<00:00, 210.24it/s]


epoch : 11/20, val detection loss = 19.574261, classification loss = 20.875269
epoch : 11/20, val acc noise = 0.9432, val acc label = 0.9771
Early stopping triggered at epoch 11
Best model was at epoch 6 with val_loss = 34.738492

Dataset split:
  - Training set: 2107357 samples
  - Validation set: 526840 samples
Final model saved
Training log saved to: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full//clique_0/mouse6_021322_natural_image_001/model_1/training_log.csv
  Clique 0, Session mouse6_021322_natural_image_001 所有重复训练完成!


In [4]:
# ============================================================
# 绘制每个clique的特征UMAP图
# ============================================================
# 每个clique生成一个PDF，包含4张UMAP图：
# 1. Noise detection GT
# 2. Noise detection predicted
# 3. Label classifier GT
# 4. Label classifier predicted

from umap import UMAP
import torch
from torch.utils import data
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages

print("="*60)
print("绘制每个clique的特征UMAP图")
print("="*60)

# 重新导入utils_clique以确保使用最新的代码定义
import importlib
import utils_clique
importlib.reload(utils_clique)
SimpleAutoSort = utils_clique.SimpleAutoSort
SimpleWaveformLoader = utils_clique.SimpleWaveformLoader

# 指定要处理的session（已经在Cell 1中定义）
# target_session_name = 'mouse6_021322_natural_image_001'  # 已在Cell 1中定义

# 只有一个session，所以session_idx是0
target_session_idx = 0

print(f"将处理session: {target_session_name} (index: {target_session_idx})")

for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"处理Clique {clique_id}")
    print(f"{'='*60}")
    
    # 只处理指定的session
    session_idx = target_session_idx
    session_name = target_session_name
    print(f"\n处理 Session {session_idx} ({session_name})...")
    
    session_data_folder = f'{combined_output_base}/clique_{clique_id}/{session_name}'
    train_data_dir = f'{session_data_folder}/train_data/'
    
    # 检查文件是否存在
    if not os.path.exists(train_data_dir):
        print(f"  警告: {train_data_dir} 不存在，跳过")
        continue
    
    # 尝试从model_1加载classification_mapping（如果不存在，尝试其他model）
    model_save_dir = None
    classification_mapping_path = None
    for repeat_idx in range(1, 6):  # 尝试model_1到model_5
        candidate_model_dir = f'{session_data_folder}/model_{repeat_idx}'
        candidate_mapping_path = f'{candidate_model_dir}/classification_mapping.pkl'
        if os.path.exists(candidate_mapping_path):
            model_save_dir = candidate_model_dir
            classification_mapping_path = candidate_mapping_path
            break
    
    if classification_mapping_path is None or not os.path.exists(classification_mapping_path):
        print(f"  警告: 未找到classification_mapping.pkl，跳过")
        continue
    
    with open(classification_mapping_path, 'rb') as f:
        classification_mapping = pickle.load(f)
    keep_id_list = classification_mapping['label_list']
    
    n_channels = 30  # 30通道
    samplepoints = 30
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    autosort_model = SimpleAutoSort(
        ch_num=n_channels,
        samplepoints=samplepoints,
        device=device,
        set_shank_id=keep_id_list,
        save_dir=model_save_dir,
        pos_weight_noise=None,
        pos_weight_label=None
    )
    
    # 加载模型权重
    noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
    label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
    
    if not os.path.exists(noise_model_path) or not os.path.exists(label_model_path):
        print(f"  警告: 模型文件不存在，跳过")
        continue
    
    autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
    autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
    autosort_model.eval()
    
    print(f"  模型已加载")
    
    # 加载训练数据
    # 需要知道shank_channel，这里使用所有通道（0到29，共30个通道）
    shank_channel = list(range(n_channels))
    dataset = SimpleWaveformLoader(train_data_dir, shank_channel, Keep_id=keep_id_list)
    dataloader = data.DataLoader(dataset, batch_size=512, shuffle=False, num_workers=0)
    
    print(f"  数据集大小: {len(dataset)}")
    
    # 提取特征和预测
    all_noise_features = []  # Features for noise classifier (intermediate_forward)
    all_label_features = []  # Features for label classifier (intermediate_forward)
    all_noise_gt = []
    all_noise_pred = []
    all_label_gt = []
    all_label_pred = []
    
    print(f"  提取特征和预测...")
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc=f"Processing batches"):
                # SimpleWaveformLoader返回顺序: Img (n_channels, window_length), GT (unit one-hot), GT_binary (noise one-hot), Img_single, channel_index
                batch_Img, batch_unit_label_onehot, batch_noise_label_onehot, batch_single, batch_channel_indices = batch_data
                batch_Img = batch_Img.to(device)  # (batch_size, n_channels, window_length)
                batch_single = batch_single.to(device)  # (batch_size, window_length)
                batch_noise_label_onehot = batch_noise_label_onehot.to(device)
                batch_unit_label_onehot = batch_unit_label_onehot.to(device)
                batch_channel_indices = batch_channel_indices.to(device) if isinstance(batch_channel_indices, torch.Tensor) else torch.tensor(batch_channel_indices, device=device)
                
                # Flatten batch_Img to (batch_size, n_channels * window_length) for _prepare_input
                batch_size = batch_Img.shape[0]
                batch_multi = batch_Img.view(batch_size, -1)  # (batch_size, n_channels * window_length)
                
                # Prepare input: codes will be (batch_size, n_channels + 2, window_length)
                codes = autosort_model._prepare_input(batch_multi, batch_single, batch_channel_indices)
                
                # Noise classifier
                noise_features = autosort_model.clsfier_noise.intermediate_forward(codes)
                noise_output = autosort_model.clsfier_noise(codes)
                noise_pred = torch.argmax(noise_output, dim=1)  # (batch_size,)
                
                # Label classifier
                label_features = autosort_model.clsfier_label.intermediate_forward(codes)
                label_output = autosort_model.clsfier_label(codes)
                label_pred = torch.argmax(label_output, dim=1)  # (batch_size,)
                
                # 将one-hot标签转换为类别索引
                # batch_noise_label_onehot: (batch_size, 2) [noise, spike] -> 0=noise, 1=spike
                noise_gt = torch.argmax(batch_noise_label_onehot, dim=1)  # (batch_size,)
                # batch_unit_label_onehot: (batch_size, n_units) -> label index (如果全为0则label=-1表示noise)
                unit_label_gt = torch.argmax(batch_unit_label_onehot, dim=1)  # (batch_size,)
                # 如果one-hot全为0（即不在任何unit中），则argmax会返回0，需要检查是否真的是valid unit
                # 可以通过检查max值来判断：如果max值为0，则表示不是valid unit
                unit_label_valid = torch.max(batch_unit_label_onehot, dim=1)[0] > 0  # (batch_size,)
                unit_label_gt = torch.where(unit_label_valid, unit_label_gt, torch.tensor(-1, device=device))
                
                # 保存特征和标签
                all_noise_features.append(noise_features.cpu().numpy())
                all_label_features.append(label_features.cpu().numpy())
                all_noise_gt.append(noise_gt.cpu().numpy())
                all_noise_pred.append(noise_pred.cpu().numpy())
                all_label_gt.append(unit_label_gt.cpu().numpy())
                all_label_pred.append(label_pred.cpu().numpy())
        
        # 合并所有batch
        all_noise_features = np.concatenate(all_noise_features, axis=0)  # (n_samples, 30)
        all_label_features = np.concatenate(all_label_features, axis=0)  # (n_samples, 30)
        all_noise_gt = np.concatenate(all_noise_gt, axis=0)  # (n_samples,)
        all_noise_pred = np.concatenate(all_noise_pred, axis=0)  # (n_samples,)
        all_label_gt = np.concatenate(all_label_gt, axis=0)  # (n_samples,)
        all_label_pred = np.concatenate(all_label_pred, axis=0)  # (n_samples,)
        
        print(f"  特征提取完成:")
        print(f"    - Noise features shape: {all_noise_features.shape}")
        print(f"    - Label features shape: {all_label_features.shape}")
        print(f"    - Noise GT: {np.unique(all_noise_gt)}")
        print(f"    - Noise Pred: {np.unique(all_noise_pred)}")
        print(f"    - Label GT unique count: {len(np.unique(all_label_gt[all_label_gt >= 0]))}")
        print(f"    - Label Pred unique count: {len(np.unique(all_label_pred))}")
        
        # 限制样本数量以提高UMAP计算速度（如果数据太多）
        # 对于noise的UMAP，从全部数据中采样
        max_samples_for_noise_umap = 50000
        if len(all_noise_features) > max_samples_for_noise_umap:
            print(f"  数据量较大，随机采样 {max_samples_for_noise_umap} 个样本用于Noise UMAP")
            noise_indices = np.random.choice(len(all_noise_features), max_samples_for_noise_umap, replace=False)
            noise_features_for_umap = all_noise_features[noise_indices]
            noise_gt_for_umap = all_noise_gt[noise_indices]
            noise_pred_for_umap = all_noise_pred[noise_indices]
        else:
            noise_features_for_umap = all_noise_features
            noise_gt_for_umap = all_noise_gt
            noise_pred_for_umap = all_noise_pred
        
        # 对于label的UMAP，先从全部数据中筛选出spike（label >= 0），然后采样5000个点
        max_samples_for_label_umap = 30000
        spike_mask = all_label_gt >= 0  # 筛选出spike
        spike_indices = np.where(spike_mask)[0]
        
        if len(spike_indices) > max_samples_for_label_umap:
            print(f"  从 {len(spike_indices)} 个spike中随机采样 {max_samples_for_label_umap} 个样本用于Label UMAP")
            selected_spike_indices = np.random.choice(len(spike_indices), max_samples_for_label_umap, replace=False)
            label_indices = spike_indices[selected_spike_indices]
        else:
            print(f"  使用全部 {len(spike_indices)} 个spike用于Label UMAP")
            label_indices = spike_indices
        
        label_features_for_umap = all_label_features[label_indices]
        label_gt_for_umap = all_label_gt[label_indices]
        label_pred_for_umap = all_label_pred[label_indices]
        
        # 同时需要对应的noise预测结果，用于绘制label predicted图
        noise_pred_for_label_umap = all_noise_pred[label_indices]
        
        # UMAP降维
        print(f"  进行UMAP降维...")
        umap_noise = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
        umap_label = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
        
        noise_features_2d = umap_noise.fit_transform(noise_features_for_umap)
        label_features_2d = umap_label.fit_transform(label_features_for_umap)
        
        # 保存PDF
        pdf_path = f'{model_save_dir}/umap_visualization_clique_{clique_id}_{session_name}.pdf'
        print(f"  保存UMAP图到: {pdf_path}")
        
        with PdfPages(pdf_path) as pdf:
            # 1. Noise detection GT
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            unique_labels = sorted(np.unique(noise_gt_for_umap))
            colors = ['lightgrey', 'orange']
            label_names = ['Noise', 'Spike']  # 通常0=noise, 1=spike
            
            for i, label in enumerate(unique_labels):
                mask = noise_gt_for_umap == label
                label_name = label_names[int(label)] if int(label) < len(label_names) else f'Class {int(label)}'
                ax.scatter(noise_features_2d[mask, 0], noise_features_2d[mask, 1], 
                          c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            # 2. Noise detection Predicted
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            unique_labels = sorted(np.unique(noise_pred_for_umap))
            colors = ['lightgrey', 'orange']
            
            for i, label in enumerate(unique_labels):
                mask = noise_pred_for_umap == label
                label_name = label_names[int(label)] if int(label) < len(label_names) else f'Class {int(label)}'
                ax.scatter(noise_features_2d[mask, 0], noise_features_2d[mask, 1], 
                          c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            if len(label_gt_for_umap) > 0:
                valid_features = label_features_2d
                valid_labels = label_gt_for_umap
                
                unique_labels = sorted(np.unique(valid_labels))
                colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = valid_labels == label
                    # 将label索引映射回unit ID
                    if int(label) < len(keep_id_list):
                        unit_id = keep_id_list[int(label)]
                        label_name = f'Unit {unit_id}'
                    else:
                        label_name = f'Label {int(label)}'
                    ax.scatter(valid_features[mask, 0], valid_features[mask, 1], 
                              c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            if len(label_pred_for_umap) > 0:
                valid_features = label_features_2d
                valid_labels = label_pred_for_umap
                
                unique_labels = sorted(np.unique(valid_labels))
                colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = valid_labels == label
                    # 将label索引映射回unit ID
                    if int(label) < len(keep_id_list):
                        unit_id = keep_id_list[int(label)]
                        label_name = f'Unit {unit_id}'
                    else:
                        label_name = f'Label {int(label)}'
                    ax.scatter(valid_features[mask, 0], valid_features[mask, 1], 
                              c=[colors[i]], label=label_name, alpha=1, s=1)
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        
        print(f"  PDF已保存: {pdf_path}")

print(f"\n{'='*60}")
print("所有UMAP图绘制完成")
print(f"{'='*60}")


绘制每个clique的特征UMAP图
将处理session: mouse6_021322_natural_image_001 (index: 0)

处理Clique 0

处理 Session 0 (mouse6_021322_natural_image_001)...
  模型已加载
Dataset loaded:
  - Total samples: 2634197
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 39
  - Noise samples: 1601542.0
  - Non-noise samples: 1032655.0
  数据集大小: 2634197
  提取特征和预测...


Processing batches: 100%|██████████| 5145/5145 [00:17<00:00, 286.70it/s]


  特征提取完成:
    - Noise features shape: (2634197, 30)
    - Label features shape: (2634197, 30)
    - Noise GT: [0 1]
    - Noise Pred: [0 1]
    - Label GT unique count: 39
    - Label Pred unique count: 39
  数据量较大，随机采样 50000 个样本用于Noise UMAP
  从 1032655 个spike中随机采样 30000 个样本用于Label UMAP
  进行UMAP降维...
  保存UMAP图到: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full//clique_0/mouse6_021322_natural_image_001/model_1/umap_visualization_clique_0_mouse6_021322_natural_image_001.pdf
  PDF已保存: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full//clique_0/mouse6_021322_natural_image_001/model_1/umap_visualization_clique_0_mouse6_021322_natural_image_001.pdf

所有UMAP图绘制完成
